# Drawing Recognition Model

Train digit + letter + shape recognition. Export to TensorFlow.js for Next.js deployment.

In [ ]:
# Cell 1 — Imports and GPU/MPS Setup
# Configure device BEFORE importing TensorFlow (for Apple Silicon Metal)
import os
import random

# Apple Silicon: enable Metal plugin for GPU acceleration
if os.environ.get("TF_METAL") is None:
    os.environ["TF_METAL"] = "1"

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import sklearn
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import albumentations as A
from tqdm import tqdm
import json
import time
import tensorflow_datasets as tfds

# --- Device detection and report ---
print("=" * 60)
print("ENVIRONMENT REPORT")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Keras version:     {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")
# Check for Metal (Apple) — may show as GPU on Mac
all_devices = tf.config.list_physical_devices()
print(f"\nDevices: {[d.device_type for d in all_devices]}")
if gpus:
    for gpu in gpus:
        print(f"  GPU: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
else:
    print("  No GPU found — using CPU (or Metal on Apple Silicon)")
print("=" * 60)

# --- Reproducibility: set all seeds to 42 ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# TF deterministic behavior (may slow training slightly)
tf.config.experimental.enable_op_determinism()

print("Random seeds set to 42. Ready for data loading.")